# Atividade — IDHM por Unidade da Federação

Nesta atividade vamos:

1. Abrir o `Tabela4.csv` com `;` e vírgula decimal;
2. Limpar colunas vazias;
3. Ordenar os estados pelo maior IDH em 2024;
4. Descobrir qual estado teve a maior melhora entre 1991 e 2024;
5. Verificar se algum estado teve piora;
6. Transformar a tabela do formato largo para o formato longo usando `melt`;
7. Plotar somente Minas Gerais;
8. Plotar a evolução do IDH de todos os estados.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Abrir o CSV
path = "Tabela4.csv"

raw = pd.read_csv(
    path,
    sep=";",
    decimal=",",
    encoding="latin1",
    header=None
)

raw.head()


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Limpar colunas totalmente vazias
raw = raw.dropna(axis=1, how="all")

# A segunda linha contém os nomes das colunas
df = raw.iloc[2:].copy()
headers = raw.iloc[1].tolist()

# Normalizar os nomes das colunas
novos_headers = []
for h in headers:
    s = str(h).strip()
    try:
        novos_headers.append(str(int(float(s))))
    except ValueError:
        novos_headers.append(s)

df.columns = novos_headers

df.head()


In [ ]:
# Identificar as colunas dos anos
anos = [c for c in df.columns if str(c).isdigit()]
print(anos)

# Converter os valores dos anos para números
for c in anos:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df.head()


## 1. Estados ordenados pelo maior IDH em 2024

In [ ]:
ranking_2024 = df.sort_values("2024", ascending=False)

ranking_2024[["Sigla", "Estado", "2024"]]


## 2. Maior melhora de IDH entre 1991 e 2024

In [ ]:
df["Melhora_1991_2024"] = df["2024"] - df["1991"]

maior_melhora = df.loc[
    df["Melhora_1991_2024"].idxmax()
]

maior_melhora[
    ["Sigla", "Estado", "1991", "2024", "Melhora_1991_2024"]
]


## 3. Existe algum estado em que o IDH piorou?

In [ ]:
pioraram = df[df["Melhora_1991_2024"] < 0]

if pioraram.empty:
    print("Não. Nenhum estado apresentou queda entre 1991 e 2024.")
else:
    display(
        pioraram[
            ["Sigla", "Estado", "1991", "2024", "Melhora_1991_2024"]
        ]
    )


## 4. Transformação para formato longo com `melt`

In [ ]:
id_vars = [c for c in df.columns if c not in anos]

df_longo = df.melt(
    id_vars=id_vars,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH",
)

df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")

df_longo.head()


## 5. Gráfico somente de Minas Gerais

In [ ]:
mg = df_longo[df_longo["Sigla"] == "MG"].sort_values("Ano")

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    mg["Ano"],
    mg["IDH"],
    marker="o",
    linewidth=2
)

ax.set_title("Evolução do IDH de Minas Gerais (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


## 6. Evolução do IDH de cada estado

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")

    ax.plot(
        grupo["Ano"],
        grupo["IDH"],
        marker="o",
        markersize=3,
        linewidth=1.5,
        label=sigla
    )

ax.set_title("Evolução do IDH por estado (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)

ax.legend(
    ncol=3,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    title="UF"
)

fig.tight_layout()
plt.show()


## 7. Exportar o resultado para CSV

In [ ]:
# Exporta a tabela em formato longo
df_longo.to_csv(
    "IDHM_resultado.csv",
    sep=";",
    decimal=",",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo IDHM_resultado.csv salvo com sucesso.")
